In [14]:
from pathlib import Path
import re
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w

data_dir = Path("/Users/jameszhao/Documents/Programs/IMC-Prosperity-3/Level5/r5data")      # change if your CSVs live elsewhere

In [19]:
price_files = sorted(data_dir.rglob("prices_round_5_day_*.csv"))

price_frames = []
for p in price_files:
    day = int(re.search(r"day_(\d+)", p.stem).group(1))
    df  = (
        pd.read_csv(p)
        .assign(day=day)
        .loc[:, ["day", "timestamp", "product", "bid_price_1", "ask_price_1"]]
    )
    price_frames.append(df)

prices = pd.concat(price_frames, ignore_index=True)

In [20]:
trade_files = sorted(data_dir.rglob("trades_round_5_day_*.csv"))

trade_frames = []
for t in trade_files:
    day = int(re.search(r"day_(\d+)", t.stem).group(1))
    df  = (
        pd.read_csv(t)
        .assign(day=day)
        .rename(columns={"symbol": "product"})      # match the prices file
    )
    trade_frames.append(df)

trades = pd.concat(trade_frames, ignore_index=True)

In [17]:
def make_fig(product):
    df_p = prices.query("product == @product")
    tr_p = trades.query("product == @product")

    fig = go.Figure()

    # -- bid / ask lines ----------------------------------------------------
    for d in sorted(df_p.day.unique()):
        block = df_p[df_p.day == d]
        fig.add_trace(go.Scatter(
            x=block.timestamp, y=block.bid_price_1,
            name=f"Day {d} best bid", mode="lines"
        ))
        fig.add_trace(go.Scatter(
            x=block.timestamp, y=block.ask_price_1,
            name=f"Day {d} best ask", mode="lines", line=dict(dash="dot")
        ))

    # -- trade markers with hover tooltips ----------------------------------
    fig.add_trace(go.Scatter(
        x=tr_p.timestamp,
        y=tr_p.price,
        mode="markers",
        name="Trades",
        marker=dict(
            symbol=[
                "triangle-up" if r.buyer < r.seller else "triangle-down"
                for r in tr_p.itertuples()
            ],
            size=10,
            opacity=0.8,
        ),
        customdata=tr_p[["buyer", "seller", "quantity"]],
        hovertemplate=(
            "<b>%{customdata[0]}</b> → <b>%{customdata[1]}</b><br>"
            "qty: %{customdata[2]}<br>"
            "price: %{y}"
            "<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=product,
        xaxis_title="Step",
        yaxis_title="Price",
        hovermode="closest",
        height=500,
        legend_title=None,
    )
    return fig

In [25]:
def make_fig(product: str) -> go.Figure:
    df_p = prices.query("product == @product")
    tr_p = trades.query("product == @product")

    fig = go.Figure()

    # -- 3a. best‑bid / best‑ask lines for each day ----------
    for d in sorted(df_p.day.unique()):
        block = df_p[df_p.day == d]

        fig.add_trace(go.Scatter(
            x=block.timestamp, y=block.bid_price_1,
            name=f"Day {d} best bid",
            mode="lines"
        ))

        fig.add_trace(go.Scatter(
            x=block.timestamp, y=block.ask_price_1,
            name=f"Day {d} best ask",
            mode="lines",
            line=dict(dash="dot")
        ))

        fig.add_trace(go.Scatter(
        x       = tr_p.timestamp,
        y       = tr_p.price,
        mode    = "markers",
        name    = "Trades",
        marker  = dict(
            symbol=[
                "triangle-up" if r.buyer < r.seller else "triangle-down"
                for r in tr_p.itertuples()
            ],
            size     = 10,
            opacity  = 0.8,
        ),
        customdata   = tr_p[["buyer", "seller", "quantity"]],
        hovertemplate=(
            "<b>%{customdata[0]}</b> → <b>%{customdata[1]}</b><br>"
            "qty: %{customdata[2]}<br>"
            "price: %{y}"
            "<extra></extra>"
        ),
    ))

    fig.update_layout(
        title         = product,
        xaxis_title   = "Step",
        yaxis_title   = "Price",
        hovermode     = "closest",
        height        = 500,
        legend_title  = None
    )
    return fig


In [ ]:
@w.interact(product=sorted(prices["product"].unique()))
def show_product(product):
    make_fig(product).show()